In [2]:
from copy import deepcopy

Solve the n queens problem as a CSP problem with forward checking and minimum remaining values heuristics

In [ ]:
class n_queens_CSP:
    def __init__(self,size):
        self.size = min(size, 20)
        self.domain = [i for i in range(self.size)]
        self.domain_all = {i:deepcopy(self.domain) for i in range(self.size)} # each var and its domain
        self.unassigned = [i for i in range(self.size)]
        self.assigned   = {} # dictionary of variable:value for all assigned variables so far
    
    def check_constraints(self, new_var, new_value):
        # check the last key:value assignment does not conflict with previous assignments
        # assumes that previous assignments are not in conflict
        if len(self.assigned) == 0: # first assignment
            return True
        
        # check rows
        if new_value in self.assigned.values():
            print("return 1")
            return False
        # check cols - already taken care
        # check main diagonal difference between index row and column are the same
        diffs = [key - value for key, value in self.assigned.items()]
        if new_var - new_value in diffs:
            print("return 2")
            return False
        # check other diagonal = sum between index row and column are the same
        sums = [key + value for key, value in self.assigned.items()]
        if new_var + new_value in sums:
            print("return 3")
            return False
        
        print("True end")
        return True
    
    def backtracking(self):
        pass

    # select the next unassigned variable:
    # 1. select the var(s) with the minimum remaining values
    # 2. if more than one var, select the most constrained variable (any queen)
    def select_unassigned(self):
        pass
    
    # select the order of values for variable var
    # from the least constraining one 
    def select_value_order(self, var):
        pass

    # update the domain of all other variables after a new assignment
    # changed self.domain_all
    def update_domain(self, var, val):
        pass

    def forward_checking(self):
        if len(self.unassigned) == 0: # you found a solution
            self.print_solution()
            return True
        
        new_var      = self.select_unassigned() # heuristic: chose the inner columns before the first and last column
        order_values = self.select_value_order(new_var) # heuristic: bottom and top rows first

        self.unassigned.remove(new_var)
        copy_domain = deepcopy(self.domain_all)
        for new_val in order_values:
            self.domain_all = deepcopy(copy_domain) # reset the domain for all variables before each new assignment
            
            if self.check_constraints(new_var, new_val): # check if assignment is valid
                self.assigned[new_var] = new_val # add assignment    
            else:
                continue
            # forward checking the domain for all remaining variables
            # changes domain_all
            self.update_domain(new_var, new_val)
            if self.check_domain() == True: # returns True if all remaining variables have at least one valid value
                if self.forward_checking(): # recursive call
                    return True
           
        # remove assignment
        self.assigned.pop(new_var)     
        self.unassigned.append(new_var)
        return False # backtrack

    # implement local search for n-queens
    # start with a random assignment of queens to rows, then 
    # iteratively move a queen to reduce the number of conflicts 
    # until a solution is found or the best solution if 
    # a maximum number of iterations is reached
    def local_search(self):
        self.assigned = {i: self.domain[i] for i in range(self.size)} # start with a random assignment of queens to rows
        max_iterations = 1000
        best_solution = self.assigned
        self.min_conflicts = self.count_conflicts()

        for _ in range(max_iterations):
            conflicts = self.count_conflicts()
            if conflicts == 0:
                self.min_conflicts = 0
                self.print_solution()
                return True
            elif conflicts < self.min_conflicts:
                self.min_conflicts = conflicts
                best_solution = self.assigned.copy()
            # find the queen with the most conflicts and move it to a new position
            queen_to_move = self.find_most_conflicted_queen()
            new_position  = self.find_best_position(queen_to_move)
            self.assigned[queen_to_move] = new_position
        
        self.assigned = best_solution
        self.print_solution()
        return False # no solution found within max iterations
    
    def count_conflicts(self):
        conflicts = 0
        for var1, val1 in self.assigned.items():
            for var2, val2 in self.assigned.items():
                if var1 < var2: # avoid double counting
                    if val1 == val2: # same row
                        conflicts += 1
                    elif abs(var1 - var2) == abs(val1 - val2): # same diagonal
                        conflicts += 1
        return conflicts
    
    def find_most_conflicted_queen(self):
        max_conflicts = -1
        queen_to_move = None
        for var in self.assigned:
            conflicts = self.count_conflicts_for_queen(var)
            if conflicts > max_conflicts:
                max_conflicts = conflicts
                queen_to_move = var
        return queen_to_move
    
    def count_conflicts_for_queen(self, queen): 
        conflicts = 0
        val_queen = self.assigned[queen]
        for var2, val2 in self.assigned.items():
            if var2 != queen:
                if val_queen == val2: # same row
                    conflicts += 1
                elif abs(queen - var2) == abs(val_queen - val2): # same diagonal
                    conflicts += 1
        return conflicts
    
    def find_best_position(self, queen):
        min_conflicts = float('inf')
        best_position = None
        for new_position in self.domain:
            original_position = self.assigned[queen]
            self.assigned[queen] = new_position
            conflicts = self.count_conflicts_for_queen(queen)
            if conflicts < min_conflicts:
                min_conflicts = conflicts
                best_position = new_position
            self.assigned[queen] = original_position # reset to original position
        return best_position
    
    def print_solution(self, solution=None):
        if solution is None:
            solution = self.assigned
        print("Solution:")
        for i in range(self.size):
            row = ['Q' if solution.get(j) == i else '.' for j in range(self.size)]
            print(' '.join(row))   
        print("\n Conflicts: ", self.min_conflicts)
         

In [47]:
queens4 = n_queens_CSP(4)

queens4.assigned[0] = 2
queens4.unassigned.remove(0)
print(queens4.assigned, queens4.unassigned)
print(list(queens4.assigned.values()).count(0))
print(queens4.check_constraints(1, 2))

{0: 2} [1, 2, 3]
0
return 1
False
